In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets , transforms
from torch.utils.data import DataLoader

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                   
 
            nn.Conv2d(32, 64, 3, padding=1),   
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                   
 
            nn.Conv2d(64, 128, 3, padding=1),  
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )
 
    def forward(self, x):
        return self.classifier(self.features(x))

In [6]:
WEIGHTS_PATH = "mnist_cnn.pth"
EPOCHS = 5
BATCH_SIZE =128
LR=1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

In [ ]:
def train():
    print(f"[INFO] Device: {DEVICE}")
 
    tf_train = transforms.Compose([
        transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    tf_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
 
    train_ds = datasets.MNIST("./data", train=True,  download=True, transform=tf_train)
    val_ds   = datasets.MNIST("./data", train=False, download=True, transform=tf_val)
 
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=256,        shuffle=False, num_workers=0)
 
    model = DigitCNN().to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=LR)
    sched = optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    crit  = nn.CrossEntropyLoss()
 
    best_acc = 0.0
 
    for epoch in range(1, EPOCHS + 1):
       
        model.train()
        running_loss = 0.0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs), labels)
            loss.backward()
            opt.step()
            running_loss += loss.item()
        sched.step()
 
       
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += labels.size(0)
 
        acc = 100 * correct / total
        print(f"  Epoch {epoch}/{EPOCHS} | loss: {running_loss/len(train_dl):.4f} | val acc: {acc:.2f}%")
 
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), WEIGHTS_PATH)
            print(f"  ✓ Saved best model → {WEIGHTS_PATH}  (acc={acc:.2f}%)")
 
    print(f"\n[DONE] Best val accuracy: {best_acc:.2f}%")
    print(f"[DONE] Weights saved to:  {WEIGHTS_PATH}")
 
 
if __name__ == "__main__":
    train()

[INFO] Device: cuda


100%|██████████| 9.91M/9.91M [00:02<00:00, 4.93MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.37MB/s]


  Epoch 1/5 | loss: 0.2611 | val acc: 98.26%
  ✓ Saved best model → mnist_cnn.pth  (acc=98.26%)
  Epoch 2/5 | loss: 0.1182 | val acc: 97.72%
  Epoch 3/5 | loss: 0.0748 | val acc: 99.26%
  ✓ Saved best model → mnist_cnn.pth  (acc=99.26%)
  Epoch 4/5 | loss: 0.0678 | val acc: 99.32%
  ✓ Saved best model → mnist_cnn.pth  (acc=99.32%)
  Epoch 5/5 | loss: 0.0531 | val acc: 99.40%
  ✓ Saved best model → mnist_cnn.pth  (acc=99.40%)

[DONE] Best val accuracy: 99.40%
[DONE] Weights saved to:  mnist_cnn.pth


In [31]:
import cv2
import torch
import torch.nn as nn

img = cv2.imread("digit.png")

if img is None:
    raise Exception("digit.png not found")

Exception: digit.png not found

device(type='cuda')